In [0]:
"""
This notebook requires a Kaggle API Token, defined in 'kaggle' databricks secrets scope.
Run the following Databricks CLI commands in a safe environment to create the scope and secret:
databricks secrets create-scope kaggle
databricks secrets put-secret kaggle KAGGLE_API_TOKEN --string-value "<your_token>"
"""

In [0]:
import os

import kagglehub
from kagglehub import KaggleDatasetAdapter

from olist_lakehouse.transformations import add_metadata

os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get("kaggle", "KAGGLE_API_TOKEN")

In [0]:
dataset = dbutils.widgets.get("dataset")
print("-" * 50)
print("Starting dataset download: ", dataset)
pandas_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    dataset,
)
print("Downloaded successfully!")
print("Adding metadata...")
spark_df = (
    spark.createDataFrame(pandas_df)
    .transform(add_metadata)
)
print("Writing to catalog delta table: ", dataset.replace(".csv", ""))
spark_df.write.mode("overwrite").saveAsTable(dataset.replace(".csv", ""))